## Bài 1


In [ ]:
"""Bài 1: vẽ cây năm lá và cây sâu tối đa 2."""

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree


ROOT = Path.cwd()
FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / "data" / "khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

tree_five_leaves = DecisionTreeClassifier(max_leaf_nodes=5, random_state=0).fit(
    X_train, y_train
)
plt.figure(figsize=(18, 9))
plot_tree(
    tree_five_leaves,
    feature_names=list(X.columns),
    class_names=["trễ hạn", "đúng hạn"],
    filled=True,
    rounded=True,
    fontsize=9,
)
plt.title("Cây quyết định có tối đa 5 lá")
plt.tight_layout()
plt.savefig(FIGURES / "cay_nam_la.png", dpi=160, bbox_inches="tight")
plt.close()

tree_depth_2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
plt.figure(figsize=(16, 8))
plot_tree(
    tree_depth_2,
    feature_names=list(X.columns),
    class_names=["trễ hạn", "đúng hạn"],
    filled=True,
    rounded=True,
    fontsize=10,
)
plt.title("Cây quyết định sâu tối đa 2")
plt.tight_layout()
plt.savefig(FIGURES / "cay_sau_2.png", dpi=160, bbox_inches="tight")
plt.close()

print("Đã lưu: figures/cay_nam_la.png")
print("Đã lưu: figures/cay_sau_2.png")
print("\nCây sâu 2:")
print(export_text(tree_depth_2, feature_names=list(X.columns), show_weights=True, decimals=3))
print("Nhận xét: hai lá bên trái đều dự đoán trễ hạn (lớp 0).")
print("Cây vẫn chia vì hai nhóm có tỷ lệ đúng/trễ hạn và độ thuần khác nhau;")
print("phép chia làm giảm Gini dù lớp chiếm đa số của cả hai lá vẫn giống nhau.")


Đã lưu: figures/cay_nam_la.png
Đã lưu: figures/cay_sau_2.png

Cây sâu 2:
|--- thu_nhap <= 15.050
|   |--- ty_le_no <= 0.615
|   |   |--- weights: [68.000, 53.000] class: 0
|   |--- ty_le_no >  0.615
|   |   |--- weights: [114.000, 14.000] class: 0
|--- thu_nhap >  15.050
|   |--- ty_le_no <= 1.020
|   |   |--- weights: [27.000, 145.000] class: 1
|   |--- ty_le_no >  1.020
|   |   |--- weights: [20.000, 9.000] class: 0

Nhận xét: hai lá bên trái đều dự đoán trễ hạn (lớp 0).
Cây vẫn chia vì hai nhóm có tỷ lệ đúng/trễ hạn và độ thuần khác nhau;
phép chia làm giảm Gini dù lớp chiếm đa số của cả hai lá vẫn giống nhau.


## Bài 2


In [ ]:
"""Bài 2: tự tìm câu hỏi gốc tốt nhất trên dữ liệu khách vay thật."""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


ROOT = Path.cwd()


def gini(labels):
    if len(labels) == 0:
        return 0.0
    p1 = np.mean(labels)
    return 1 - p1**2 - (1 - p1) ** 2


def tim_cau_hoi(X, y):
    """Thử mọi cột và ngưỡng giữa hai giá trị liên tiếp, trả về câu hỏi tốt nhất."""
    y_array = np.asarray(y)
    gini_goc = gini(y_array)
    best = {"cot": None, "nguong": None, "gini_sau_chia": np.inf, "gini_giam": -np.inf}

    for column in X.columns:
        values = X[column].to_numpy(dtype=float)
        unique_values = np.sort(np.unique(values))
        thresholds = (unique_values[:-1] + unique_values[1:]) / 2
        for threshold in thresholds:
            left = values <= threshold
            if left.all() or (~left).all():
                continue
            gini_split = (
                left.mean() * gini(y_array[left])
                + (~left).mean() * gini(y_array[~left])
            )
            decrease = gini_goc - gini_split
            if decrease > best["gini_giam"]:
                best = {
                    "cot": column,
                    "nguong": float(threshold),
                    "gini_sau_chia": float(gini_split),
                    "gini_giam": float(decrease),
                }
    return best


df = pd.read_csv(ROOT / "data" / "khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

best = tim_cau_hoi(X_train, y_train)
print(f"Cột tốt nhất       : {best['cot']}")
print(f"Ngưỡng tốt nhất    : {best['nguong']:.3f}")
print(f"Gini sau chia      : {best['gini_sau_chia']:.4f}")
print(f"Mức Gini giảm      : {best['gini_giam']:.4f}")

print("\nGiải thích:")
print("- Với mỗi cột, hàm sắp xếp các giá trị khác nhau và thử ngưỡng ở chính giữa")
print("  từng cặp giá trị liên tiếp. Các ngưỡng nằm trong cùng một khoảng chia dữ liệu như nhau.")
print("- Mỗi câu hỏi được chấm bằng Gini trung bình có trọng số của hai nhánh.")
print("- Câu hỏi làm Gini giảm nhiều nhất là thu_nhap <= 15.05, trùng câu hỏi gốc")
print("  của cây scikit-learn trong bài thực hành.")


Cột tốt nhất       : thu_nhap
Ngưỡng tốt nhất    : 15.050
Gini sau chia      : 0.3777
Mức Gini giảm      : 0.1221

Giải thích:
- Với mỗi cột, hàm sắp xếp các giá trị khác nhau và thử ngưỡng ở chính giữa
  từng cặp giá trị liên tiếp. Các ngưỡng nằm trong cùng một khoảng chia dữ liệu như nhau.
- Mỗi câu hỏi được chấm bằng Gini trung bình có trọng số của hai nhánh.
- Câu hỏi làm Gini giảm nhiều nhất là thu_nhap <= 15.05, trùng câu hỏi gốc
  của cây scikit-learn trong bài thực hành.


## Bài 3


In [ ]:
"""Bài 3: so sánh tiêu chí entropy với Gini."""

from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)


def chon_do_sau(criterion):
    scores = {
        depth: cross_val_score(
            DecisionTreeClassifier(max_depth=depth, criterion=criterion, random_state=0),
            X_train,
            y_train,
            cv=5,
        ).mean()
        for depth in range(1, 16)
    }
    best_depth = max(scores, key=scores.get)
    model = DecisionTreeClassifier(
        max_depth=best_depth, criterion=criterion, random_state=0
    ).fit(X_train, y_train)
    test_accuracy = accuracy_score(y_test, model.predict(X_test))
    return best_depth, scores[best_depth], test_accuracy


gini_result = chon_do_sau("gini")
entropy_result = chon_do_sau("entropy")

print(f"{'Tiêu chí':<10} {'độ sâu':>8} {'CV':>10} {'test':>10}")
print(f"{'Gini':<10} {gini_result[0]:8d} {gini_result[1]:10.4f} {gini_result[2]:10.4f}")
print(f"{'Entropy':<10} {entropy_result[0]:8d} {entropy_result[1]:10.4f} {entropy_result[2]:10.4f}")
print(
    f"\nNhận xét: đổi sang entropy làm accuracy kiểm tra thay đổi "
    f"{entropy_result[2] - gini_result[2]:+.4f}; mức thay đổi này không lớn."
)


Tiêu chí     độ sâu         CV       test
Gini              5     0.8333     0.8533
Entropy           5     0.8222     0.8467

Nhận xét: đổi sang entropy làm accuracy kiểm tra thay đổi -0.0067; mức thay đổi này không lớn.


## Bài 4


In [ ]:
"""Bài 4: cắt tỉa bằng số khách tối thiểu ở lá."""

from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

leaf_sizes = [1, 2, 5, 10, 20, 40]
scores = {}
print(f"{'min_samples_leaf':>18} {'CV trung bình':>14}")
for leaf_size in leaf_sizes:
    model = DecisionTreeClassifier(min_samples_leaf=leaf_size, random_state=0)
    scores[leaf_size] = cross_val_score(model, X_train, y_train, cv=5).mean()
    print(f"{leaf_size:18d} {scores[leaf_size]:14.4f}")

best_leaf_size = max(scores, key=scores.get)
best_score = scores[best_leaf_size]
ties = [size for size in leaf_sizes if abs(scores[size] - best_score) < 1e-12]
best_model = DecisionTreeClassifier(
    min_samples_leaf=best_leaf_size, random_state=0
).fit(X_train, y_train)
leaf_test_accuracy = accuracy_score(y_test, best_model.predict(X_test))

depth_5 = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_train, y_train)
depth_5_accuracy = accuracy_score(y_test, depth_5.predict(X_test))

print(f"\nGiá trị có điểm CV cao nhất: {', '.join(map(str, ties))}")
print(f"Chọn min_samples_leaf = {best_leaf_size} (giá trị nhỏ hơn khi hòa điểm)")
print(f"Accuracy cách cắt theo số khách ở lá: {leaf_test_accuracy:.4f}")
print(f"Accuracy cây sâu tối đa 5          : {depth_5_accuracy:.4f}")
if leaf_test_accuracy > depth_5_accuracy:
    print("Cắt theo min_samples_leaf tốt hơn trên tập kiểm tra của bộ dữ liệu này.")
elif leaf_test_accuracy < depth_5_accuracy:
    print("Giới hạn max_depth=5 tốt hơn trên tập kiểm tra của bộ dữ liệu này.")
else:
    print("Hai cách cắt tỉa cho cùng accuracy trên tập kiểm tra.")


  min_samples_leaf  CV trung bình
                 1         0.7733
                 2         0.7822
                 5         0.8067
                10         0.8089
                20         0.8089
                40         0.7133

Giá trị có điểm CV cao nhất: 10, 20
Chọn min_samples_leaf = 10 (giá trị nhỏ hơn khi hòa điểm)
Accuracy cách cắt theo số khách ở lá: 0.8467
Accuracy cây sâu tối đa 5          : 0.8533
Giới hạn max_depth=5 tốt hơn trên tập kiểm tra của bộ dữ liệu này.


## Bài 5


In [ ]:
"""Bài 5: đo độ quan trọng bằng cách xáo trộn."""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

rng = np.random.default_rng(0)
X_train_random = X_train.copy()
X_test_random = X_test.copy()
X_train_random["so_ngau_nhien"] = rng.random(len(X_train_random))
X_test_random["so_ngau_nhien"] = rng.random(len(X_test_random))

forest = RandomForestClassifier(n_estimators=200, random_state=0).fit(
    X_train_random, y_train
)

impurity_importance = pd.Series(
    forest.feature_importances_, index=X_train_random.columns
).sort_values(ascending=False)

permutation = permutation_importance(
    forest,
    X_test_random,
    y_test,
    n_repeats=20,
    random_state=0,
)
permutation_mean = pd.Series(
    permutation.importances_mean, index=X_test_random.columns
).sort_values(ascending=False)

print("Độ quan trọng do giảm Gini (feature_importances_):")
print(impurity_importance.round(4).to_string())
print("\nĐộ quan trọng xáo trộn trung bình trên tập kiểm tra:")
print(permutation_mean.round(4).to_string())

rank_impurity = impurity_importance.index.get_loc("so_ngau_nhien") + 1
rank_permutation = permutation_mean.index.get_loc("so_ngau_nhien") + 1
print(f"\nCột ngẫu nhiên xếp thứ {rank_impurity} theo giảm Gini.")
print(f"Cột ngẫu nhiên xếp thứ {rank_permutation} theo permutation importance.")
print("Giải thích: feature_importances_ thiên vị cột có nhiều giá trị vì cây có nhiều ngưỡng")
print("để thử và dễ tìm được phép chia tình cờ tốt trên tập học. Permutation importance đo trên")
print("tập kiểm tra: xáo trộn cột vô dụng hầu như không làm accuracy giảm, nên thứ hạng thấp hơn.")
print("Độ quan trọng chỉ cho biết mô hình sử dụng cột, không chứng minh quan hệ nhân quả.")


Độ quan trọng do giảm Gini (feature_importances_):
thu_nhap              0.4269
ty_le_no              0.2696
so_ngau_nhien         0.1499
co_viec_on_dinh       0.0773
so_nguoi_phu_thuoc    0.0764

Độ quan trọng xáo trộn trung bình trên tập kiểm tra:
thu_nhap              0.3073
ty_le_no              0.1993
co_viec_on_dinh       0.1117
so_nguoi_phu_thuoc    0.0497
so_ngau_nhien         0.0017

Cột ngẫu nhiên xếp thứ 3 theo giảm Gini.
Cột ngẫu nhiên xếp thứ 5 theo permutation importance.
Giải thích: feature_importances_ thiên vị cột có nhiều giá trị vì cây có nhiều ngưỡng
để thử và dễ tìm được phép chia tình cờ tốt trên tập học. Permutation importance đo trên
tập kiểm tra: xáo trộn cột vô dụng hầu như không làm accuracy giảm, nên thứ hạng thấp hơn.
Độ quan trọng chỉ cho biết mô hình sử dụng cột, không chứng minh quan hệ nhân quả.


## Bài 6


In [ ]:
"""Bài 6: rừng ngẫu nhiên cho bài toán hồi quy."""

from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

forest = RandomForestRegressor(n_estimators=200, random_state=0).fit(X_train, y_train)
tree = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X_train, y_train)
linear = LinearRegression().fit(X_train, y_train)

mae_forest = mean_absolute_error(y_test, forest.predict(X_test))
mae_tree = mean_absolute_error(y_test, tree.predict(X_test))
mae_linear = mean_absolute_error(y_test, linear.predict(X_test))

print("MAE trên tập kiểm tra (tỷ đồng)")
print(f"  Rừng 200 cây: {mae_forest:.4f}")
print(f"  Cây sâu 3   : {mae_tree:.4f}")
print(f"  Đường thẳng : {mae_linear:.4f}")

house_250 = pd.DataFrame({"dien_tich": [250.0]})
prediction_250 = forest.predict(house_250)[0]
print(f"\nDự đoán của rừng cho căn 250 m2: {prediction_250:.4f} tỷ đồng")
print("Rừng không ngoại suy được. Mỗi cây đưa căn 250 m2 tới một lá cuối và dự đoán bằng")
print("trung bình giá các căn tập học trong lá đó; rừng chỉ lấy trung bình các dự đoán của 200 cây.")
print("Vì vậy dự đoán vẫn bị giới hạn bởi các mức giá đã xuất hiện trong tập học.")


MAE trên tập kiểm tra (tỷ đồng)
  Rừng 200 cây: 0.4559
  Cây sâu 3   : 0.5039
  Đường thẳng : 0.2578

Dự đoán của rừng cho căn 250 m2: 9.1973 tỷ đồng
Rừng không ngoại suy được. Mỗi cây đưa căn 250 m2 tới một lá cuối và dự đoán bằng
trung bình giá các căn tập học trong lá đó; rừng chỉ lấy trung bình các dự đoán của 200 cây.
Vì vậy dự đoán vẫn bị giới hạn bởi các mức giá đã xuất hiện trong tập học.
